# Indian Retail Data Processing & Augmentation


In [14]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import random

## 1. Data Loading & Preprocessing

In [15]:
df = pd.read_csv("online_retail_II.csv")
print(f"Retail data loaded. Shape: {df.shape}")
df.head()

Retail data loaded. Shape: (1048575, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,12/1/2009 7:45,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,12/1/2009 7:45,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,12/1/2009 7:45,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,12/1/2009 7:45,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,12/1/2009 7:45,1.25,13085.0,United Kingdom


In [16]:
indian_cities_df = pd.read_csv('indian_cities.csv')
print(f"Indian cities data loaded. Shape: {indian_cities_df.shape}")
indian_cities_df.head()

Indian cities data loaded. Shape: (297, 4)


,City,State,Latitude,Longitude
0,Mumbai,Maharashtra,19.0760,72.8777
1,Delhi,Delhi,28.7041,77.1025
2,Bangalore,Karnataka,12.9716,77.5946
3,Hyderabad,Telangana,17.3850,78.4867
4,Ahmedabad,Gujarat,23.0225,72.5714


In [17]:
df['Is_Return'] = np.where(df['Invoice'].astype(str).str.startswith('C') & (df['Quantity'] < 0), 1, 0)

In [18]:
df.dropna(subset=['Customer ID'], inplace=True)

In [19]:
on_product_codes = ['POST', 'M', 'BANK CHARGES', 'D', 'CRUK', 'PADS', 'DOT']
df = df[~df['StockCode'].isin(non_product_codes)]

### 1.1 Data Augmentation: Increase Return Rate

In [20]:
# Target Return Rate: ~12% (User requested 10-15%)
current_return_rate = df['Is_Return'].mean()
target_return_rate = 0.13

if current_return_rate < target_return_rate:
    print(f"Current Return Rate: {current_return_rate:.2%}. Augmenting to {target_return_rate:.2%}...")
    
    total_rows = len(df)
    target_returns = int(total_rows * target_return_rate)
    current_returns = df['Is_Return'].sum()
    needed_returns = target_returns - current_returns
    
    if needed_returns > 0:
        # Select random non-return indices to flip
        candidates = df[df['Is_Return'] == 0].index
        # Ensure we don't try to sample more than available (unlikely but safe)
        n_sample = min(len(candidates), int(needed_returns))
        
        flip_indices = np.random.choice(candidates, n_sample, replace=False)
        
        # Apply changes
        df.loc[flip_indices, 'Is_Return'] = 1
        
        # Make Quantity negative if it's positive (standardize returns)
        # Note: Some original non-returns might already have varying quantities, ensure they become negative
        df.loc[flip_indices, 'Quantity'] = -1 * df.loc[flip_indices, 'Quantity'].abs()
        
        # Optional: Prefix Invoice with 'C' to mimic real returns
        # using apply might be slow, vectorized approach:
        # df.loc[flip_indices, 'Invoice'] = 'C' + df.loc[flip_indices, 'Invoice'].astype(str)
        # However, Invoice format might differ. For Is_Fraud logic, Is_Return column is key.
        
        print(f"Injected {n_sample} synthetic returns.")
        print(f"New Return Rate: {df['Is_Return'].mean():.2%}")

Current Return Rate: 2.19%. Augmenting to 13.00%...
Injected 87364 synthetic returns.
New Return Rate: 13.00%


In [21]:
# Date Parsing
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Total Price (Recalculate to ensure negative quantity results in negative price)
if 'Price' in df.columns:
    df['Total_Price'] = df['Quantity'] * df['Price']
elif 'UnitPrice' in df.columns:
    df['Total_Price'] = df['Quantity'] * df['UnitPrice']

## 2. Indian Data Augmentation
Mapping customers to Indian cities based on `indian_cities.csv`.

In [22]:
df = df.drop(columns=['Country'])

In [23]:
city_records = indian_cities_df.to_dict('records')

In [24]:
customer_ids = df['Customer ID'].unique()
# Randomly assign a city dict to each customer
customer_city_map = {cid: random.choice(city_records) for cid in customer_ids}

In [25]:
mapping_df = pd.DataFrame.from_dict(customer_city_map, orient='index')
mapping_df.index.name = 'Customer ID'

In [26]:
df = df.merge(mapping_df, on='Customer ID', how='left')

## 3. Feature Engineering
Calculating risk factors: `Product_Return_Rate`, `User_Global_Return_Rate`, and `Multi_Size_Order` (Wardrobing).

In [27]:
# Product Risk: Product_Return_Rate
product_stats = df.groupby('StockCode').agg(
    Total_Sold=('Quantity', lambda x: x[x > 0].sum()),
    Total_Returned=('Quantity', lambda x: x[x < 0].abs().sum())
).reset_index()

In [28]:
product_stats['Product_Return_Rate'] = product_stats['Total_Returned'] / (product_stats['Total_Sold'] + product_stats['Total_Returned'])
product_stats['Product_Return_Rate'] = product_stats['Product_Return_Rate'].fillna(0)

In [29]:
df = df.merge(product_stats[['StockCode', 'Product_Return_Rate']], on='StockCode', how='left')

In [30]:
# Customer Risk: User_Global_Return_Rate
user_stats = df.groupby('Customer ID').agg(
    User_Total_Bought=('Quantity', lambda x: x[x > 0].sum()),
    User_Total_Returned=('Quantity', lambda x: x[x < 0].abs().sum())
).reset_index()

In [31]:
user_stats['User_Global_Return_Rate'] = user_stats['User_Total_Returned'] / (user_stats['User_Total_Bought'] + user_stats['User_Total_Returned'])
user_stats['User_Global_Return_Rate'] = user_stats['User_Global_Return_Rate'].fillna(0)

In [32]:
df = df.merge(user_stats[['Customer ID', 'User_Global_Return_Rate']], on='Customer ID', how='left')

In [33]:
# Wardrobing Indicator: Multi_Size_Order
df['StockCode_Base'] = df['StockCode'].astype(str).str.extract(r'^(\d{5})')
invoice_base_counts = df.groupby(['Invoice', 'StockCode_Base'])['StockCode'].nunique().reset_index(name='Variation_Count')
wardrobing_invoices = invoice_base_counts[invoice_base_counts['Variation_Count'] > 1]['Invoice'].unique()
df['Multi_Size_Order'] = df['Invoice'].isin(wardrobing_invoices)

In [ ]:
# Save the augmented dataset
output_file = 'online_retail_indian_augmented.csv'
df.to_csv(output_file, index=False)
print(f"Augmented Indian dataset saved to {output_file}")

Augmented Indian dataset saved to online_retail_indian_augmented1.csv
